# Data Science: Hyperparameter Tuning as Continuous Optimization

Grid search wastes budget on a fixed lattice; random search ignores the fact that nearby hyperparameters tend to have similar performance. A metaheuristic treats hyperparameter tuning as what it is: optimizing a (noisy, expensive, black-box) objective, $-\text{CV accuracy}$, over a continuous space — here, $\log_{10}(C)$ and $\log_{10}(\gamma)$ for an SVM's RBF kernel.

In [1]:
import numpy as np
from scipy.stats import loguniform
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.svm import SVC

from metaheuristics.algorithms.particle_swarm import ParticleSwarmOptimization

X, y = load_breast_cancer(return_X_y=True)

## PSO over $(\log_{10} C, \log_{10}\gamma)$

In [2]:
def svm_cv_error(params):
    C, gamma = 10 ** params[0], 10 ** params[1]
    return 1 - cross_val_score(SVC(C=C, gamma=gamma), X, y, cv=5).mean()

bounds = [(-2, 4), (-6, 1)]  # log10(C), log10(gamma)

np.random.seed(0)
result = ParticleSwarmOptimization(num_particles=12, max_iterations=12).optimize(svm_cv_error, bounds)
best_C, best_gamma = 10 ** result.best_solution[0], 10 ** result.best_solution[1]
print(f'PSO best C={best_C:.4g}, gamma={best_gamma:.4g}, CV accuracy={1 - result.best_fitness:.4f}')

PSO best C=1e+04, gamma=1.209e-06, CV accuracy=0.9543


## Baselines: grid search (fixed lattice) and random search (matched budget)

In [3]:
grid = GridSearchCV(
    SVC(), param_grid={'C': np.logspace(-2, 4, 5), 'gamma': np.logspace(-6, 1, 5)}, cv=5
).fit(X, y)
print(f'GridSearchCV      best CV accuracy={grid.best_score_:.4f}, params={grid.best_params_}')

random_search = RandomizedSearchCV(
    SVC(),
    param_distributions={'C': loguniform(1e-2, 1e4), 'gamma': loguniform(1e-6, 1e1)},
    n_iter=12 * 13,  # match PSO's roughly (iterations + 1) * particles budget
    cv=5,
    random_state=0,
).fit(X, y)
print(f'RandomizedSearchCV best CV accuracy={random_search.best_score_:.4f}, params={random_search.best_params_}')

GridSearchCV      best CV accuracy=0.9526, params={'C': 10000.0, 'gamma': 1e-06}


RandomizedSearchCV best CV accuracy=0.9578, params={'C': 420.22752076780404, 'gamma': 1.0681359144442711e-05}
